# 🧠 HuggingFace LLM Server — SillyTavern API (Google Colab)

### One-Click Setup: GPU Model Server mit SillyTavern-kompatibler API

| Feature | Details |
|---|---|
| **Model** | `intervitens/mini-magnum-12b-v1.1` (12.2B params, Mistral arch) |
| **Engine** | vLLM — OpenAI-kompatible API |
| **GPU** | NVIDIA T4/L4/A100 (auto-detect + quantization) |
| **Tunnel** | Gradio Share + Cloudflared für SillyTavern |
| **Storage** | Google Drive persistent model cache |
| **API** | `/v1/chat/completions`, `/v1/completions`, `/v1/models` |

> ⚡ **Anleitung:** Runtime → GPU auswählen → Alle Zellen ausführen → SillyTavern URL kopieren
>
> 💾 Modelle werden auf Google Drive gespeichert → kein erneuter Download beim nächsten Start!

In [ ]:
#@title ⚙️ 1. GPU Check & Install Dependencies
import subprocess, os, sys, time
from IPython.display import display, HTML, clear_output

def box(title, lines, color="#2196F3"):
    """Display a styled output box."""
    html = f'<div style="border:2px solid {color};border-radius:10px;padding:15px;margin:10px 0;background:#1a1a2e;color:#eee;font-family:monospace;">'
    html += f'<div style="font-size:16px;font-weight:bold;color:{color};margin-bottom:10px;">{title}</div>'
    for l in lines:
        html += f'<div style="margin:3px 0;font-size:13px;">{l}</div>'
    html += '</div>'
    display(HTML(html))

print("=" * 60)
print("📹 GPU CHECK")
print("=" * 60)

# Safe GPU detection with error handling
gpu_name = "No GPU"
gpu_mem_mb = 0
gpu_mem_gb = 0
gpu_available = False

try:
    gpu_info_output = !nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv,noheader
    if gpu_info_output and "NVIDIA" in gpu_info_output[0]:
        print(gpu_info_output[0])
        gpu_available = True

        gpu_name_output = !nvidia-smi --query-gpu=name --format=csv,noheader
        gpu_name = gpu_name_output[0].strip() if gpu_name_output else "Unknown GPU"

        gpu_mem_output = !nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits
        gpu_mem_mb = int(gpu_mem_output[0].strip()) if gpu_mem_output else 0
        gpu_mem_gb = gpu_mem_mb / 1024
    else:
        print("⚠️ nvidia-smi nicht verfügbar oder keine GPU erkannt.")
except Exception as e:
    print(f"⚠️ GPU-Erkennung fehlgeschlagen: {e}")
    print("   Stelle sicher, dass ein GPU-Runtime ausgewählt ist!")

os.environ["GPU_NAME"] = gpu_name
os.environ["GPU_MEM_MB"] = str(gpu_mem_mb)

if not gpu_available:
    gpu_status = "❌ Keine GPU gefunden! Gehe zu Runtime → Change runtime type → GPU"
    gpu_color = "#ff5252"
elif gpu_mem_mb < 15000:
    gpu_status = "⚠️ <16GB VRAM — Model passt evtl. nicht. Nutze A100!"
    gpu_color = "#ff5252"
elif gpu_mem_mb < 24000:
    gpu_status = "ℹ️ T4 erkannt → AWQ/BnB 4-bit Quantisierung wird genutzt"
    gpu_color = "#FFC107"
else:
    gpu_status = "✅ Genug VRAM für Half-Precision"
    gpu_color = "#4CAF50"

box("🖥️ GPU Status", [
    f"GPU: <b>{gpu_name}</b>",
    f"VRAM: <b>{gpu_mem_gb:.0f} GB</b> ({gpu_mem_mb} MB)" if gpu_available else "VRAM: <b>N/A</b>",
    f"Status: {gpu_status}",
], gpu_color)

if not gpu_available:
    raise RuntimeError("Keine GPU verfügbar. Bitte GPU-Runtime auswählen bevor du fortfährst.")

print("\n📦 INSTALLING DEPENDENCIES...")
!pip install -q vllm huggingface_hub gradio requests tqdm 2>&1 | tail -1
print("✅ Alle Dependencies installiert!")

In [ ]:
#@title 💾 2. Google Drive Mount — Persistenter Model Cache
#@markdown Modelle werden auf Google Drive gespeichert.
#@markdown Beim nächsten Start wird das Model von Drive geladen (kein Re-Download).
USE_GDRIVE = True #@param {type:"boolean"}
GDRIVE_MODEL_PATH = "/content/drive/MyDrive/LLM_Models" #@param {type:"string"}

import os
from pathlib import Path
from IPython.display import display, HTML

LOCAL_MODEL_PATH = "/root/models"

if USE_GDRIVE:
    from google.colab import drive
    print("📂 Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=False)

    os.makedirs(GDRIVE_MODEL_PATH, exist_ok=True)
    os.environ["USE_GDRIVE"] = "1"
    os.environ["GDRIVE_MODEL_PATH"] = GDRIVE_MODEL_PATH

    # Check existing models on Drive
    existing = []
    if os.path.exists(GDRIVE_MODEL_PATH):
        for d in sorted(os.listdir(GDRIVE_MODEL_PATH)):
            full = os.path.join(GDRIVE_MODEL_PATH, d)
            if os.path.isdir(full):
                try:
                    size_gb = sum(f.stat().st_size for f in Path(full).rglob('*') if f.is_file()) / (1024**3)
                    existing.append(f"📁 {d} — <b>{size_gb:.1f} GB</b>")
                except OSError:
                    existing.append(f"📁 {d} — <i>Größe unbekannt</i>")

    lines = [
        f"Mount: <b>/content/drive/MyDrive</b>",
        f"Model Cache: <b>{GDRIVE_MODEL_PATH}</b>",
    ]
    if existing:
        lines.append("<br><b>Gespeicherte Modelle:</b>")
        lines.extend(existing)
    else:
        lines.append("Noch keine Modelle auf Drive gecacht.")

    display(HTML(
        '<div style="border:2px solid #4CAF50;border-radius:10px;padding:15px;margin:10px 0;'
        'background:#1a1a2e;color:#eee;font-family:monospace;">'
        '<div style="font-size:16px;font-weight:bold;color:#4CAF50;margin-bottom:10px;">'
        '💾 Google Drive Verbunden</div>'
        + ''.join(f'<div style="margin:3px 0;font-size:13px;">{l}</div>' for l in lines)
        + '</div>'
    ))
    print("✅ Google Drive verbunden!")
else:
    os.environ["USE_GDRIVE"] = "0"
    os.makedirs(LOCAL_MODEL_PATH, exist_ok=True)
    print(f"ℹ️ Google Drive deaktiviert. Modelle werden temporär in {LOCAL_MODEL_PATH} gespeichert.")
    print("   ⚠️ Daten gehen bei Colab Disconnect verloren!")

In [ ]:
#@title 📥 3. Download Model von HuggingFace (mit Live-Status)
#@markdown Model to download:
MODEL_REPO = "intervitens/mini-magnum-12b-v1.1" #@param {type:"string"}
#@markdown HuggingFace Token (optional, für gated models):
HF_TOKEN = "" #@param {type:"string"}

import os, time, threading
from huggingface_hub import snapshot_download, HfApi
from pathlib import Path
from IPython.display import display, HTML, clear_output

# Determine storage path (Google Drive or local)
use_gdrive = os.environ.get("USE_GDRIVE", "0") == "1"
gdrive_path = os.environ.get("GDRIVE_MODEL_PATH", "/content/drive/MyDrive/LLM_Models")
model_slug = MODEL_REPO.replace('/', '--')

if use_gdrive:
    MODEL_DIR = os.path.join(gdrive_path, model_slug)
    storage_info = f"💾 Google Drive: {MODEL_DIR}"
else:
    MODEL_DIR = f"/root/models/{model_slug}"
    storage_info = f"📂 Lokal (temporär): {MODEL_DIR}"

os.environ["MODEL_REPO"] = MODEL_REPO
os.environ["MODEL_DIR"] = MODEL_DIR
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()

# ── Model Info ──
print("=" * 60)
print(f"📥 MODEL: {MODEL_REPO}")
print(f"   {storage_info}")
print("=" * 60)

total_size_gb = 0
file_count = 0
try:
    api = HfApi(token=HF_TOKEN.strip() or None)
    model_info = api.model_info(MODEL_REPO)
    siblings = model_info.siblings or []
    total_size_gb = sum(s.size for s in siblings if s.size) / (1024**3)
    file_count = len(siblings)
    print(f"\n📊 Model Info:")
    print(f"   Größe: ~{total_size_gb:.1f} GB")
    print(f"   Dateien: {file_count}")
    print(f"   Architektur: {getattr(model_info, 'library_name', '?')} / {', '.join(getattr(model_info, 'tags', [])[:5])}")
except Exception as e:
    print(f"   Model-Info nicht verfügbar: {e}")

# ── Check if already downloaded ──
config_exists = os.path.exists(os.path.join(MODEL_DIR, "config.json"))
if config_exists:
    existing_size = sum(f.stat().st_size for f in Path(MODEL_DIR).rglob('*') if f.is_file()) / (1024**3)
    print(f"\n✅ Model bereits vorhanden! ({existing_size:.1f} GB)")
    print(f"   → Überspringe Download.")
    
    display(HTML(
        '<div style="border:2px solid #4CAF50;border-radius:10px;padding:15px;margin:10px 0;background:#1a1a2e;color:#eee;font-family:monospace;">'
        '<div style="font-size:16px;font-weight:bold;color:#4CAF50;">✅ Model aus Cache geladen</div>'
        f'<div style="margin:5px 0;font-size:13px;">Pfad: <b>{MODEL_DIR}</b></div>'
        f'<div style="margin:5px 0;font-size:13px;">Größe: <b>{existing_size:.1f} GB</b></div>'
        '</div>'
    ))
else:
    # ── Download with live progress ──
    print(f"\n⬇️ Starte Download...")
    start_time = time.time()

    # Background download progress monitor
    download_active = True
    def monitor_download():
        """Monitor download directory size in background."""
        while download_active:
            try:
                if os.path.exists(MODEL_DIR):
                    current = sum(f.stat().st_size for f in Path(MODEL_DIR).rglob('*') if f.is_file()) / (1024**3)
                    elapsed = time.time() - start_time
                    speed = current / elapsed if elapsed > 0 else 0
                    pct = (current / total_size_gb * 100) if total_size_gb > 0 else 0
                    eta = (total_size_gb - current) / speed if speed > 0 else 0

                    bar_len = 30
                    filled = int(bar_len * pct / 100)
                    bar = "█" * filled + "░" * (bar_len - filled)

                    print(f"\r   [{bar}] {pct:5.1f}% | {current:.1f}/{total_size_gb:.1f} GB | {speed*1024:.0f} MB/s | ETA: {eta:.0f}s   ", end="", flush=True)
            except:
                pass
            time.sleep(3)

    monitor = threading.Thread(target=monitor_download, daemon=True)
    monitor.start()

    # Actual download
    local_path = snapshot_download(
        repo_id=MODEL_REPO,
        local_dir=MODEL_DIR,
        token=HF_TOKEN.strip() or None,
        resume_download=True,
    )
    download_active = False
    elapsed = time.time() - start_time

    final_size = sum(f.stat().st_size for f in Path(local_path).rglob('*') if f.is_file()) / (1024**3)
    speed_avg = final_size / elapsed if elapsed > 0 else 0

    print(f"\n\n{'=' * 60}")

    # Show downloaded files
    file_list = []
    for f in sorted(Path(local_path).rglob("*")):
        if f.is_file() and not f.name.startswith("."):
            size_mb = f.stat().st_size / (1024**2)
            file_list.append(f"📄 {f.name} — {size_mb:.0f} MB")

    display(HTML(
        '<div style="border:2px solid #4CAF50;border-radius:10px;padding:15px;margin:10px 0;background:#1a1a2e;color:#eee;font-family:monospace;">'
        '<div style="font-size:16px;font-weight:bold;color:#4CAF50;">✅ Download Abgeschlossen!</div>'
        f'<div style="margin:8px 0;font-size:13px;">Model: <b>{MODEL_REPO}</b></div>'
        f'<div style="margin:3px 0;font-size:13px;">Größe: <b>{final_size:.1f} GB</b> | Zeit: <b>{elapsed:.0f}s</b> | Speed: <b>{speed_avg*1024:.0f} MB/s</b></div>'
        f'<div style="margin:3px 0;font-size:13px;">Pfad: <b>{local_path}</b></div>'
        f'<div style="margin:3px 0;font-size:13px;">Storage: <b>{"Google Drive ✅" if use_gdrive else "Lokal (temporär)"}</b></div>'
        '<hr style="border-color:#333;">'
        '<div style="font-size:12px;color:#aaa;">' + '<br>'.join(file_list[:15]) + '</div>'
        '</div>'
    ))

In [ ]:
#@title 🚀 4. Start vLLM Server (Background, NVIDIA GPU)
#@markdown Server Einstellungen:
MAX_MODEL_LEN = 4096 #@param {type:"integer"}
TENSOR_PARALLEL = 1 #@param {type:"integer"}
GPU_MEMORY_UTILIZATION = 0.92 #@param {type:"number"}
QUANTIZATION = "auto" #@param ["auto", "none", "awq", "gptq", "bitsandbytes"]
#@markdown `auto` = erkennt automatisch ob Quantisierung nötig ist

import subprocess, os, time, requests, json, threading
from IPython.display import display, HTML

MODEL_DIR = os.environ.get("MODEL_DIR", "/root/models/intervitens--mini-magnum-12b-v1.1")
MODEL_REPO = os.environ.get("MODEL_REPO", "intervitens/mini-magnum-12b-v1.1")
gpu_mem_mb = int(os.environ.get("GPU_MEM_MB", "16000"))

# ── Auto-detect quantization ──
quant_args = []
quant_label = "none"
if QUANTIZATION == "auto":
    config_path = os.path.join(MODEL_DIR, "config.json")
    quant_method = None
    if os.path.exists(config_path):
        with open(config_path) as f:
            cfg = json.load(f)
            qc = cfg.get("quantization_config", {})
            quant_method = qc.get("quant_method", None)

    if quant_method:
        quant_label = quant_method
        quant_args = ["--quantization", quant_method]
    elif gpu_mem_mb < 24000:
        quant_label = "bitsandbytes (4-bit)"
        quant_args = ["--quantization", "bitsandbytes", "--load-format", "bitsandbytes"]
    else:
        quant_label = "half-precision (fp16)"
elif QUANTIZATION != "none":
    quant_label = QUANTIZATION
    quant_args = ["--quantization", QUANTIZATION]

# ── Build vLLM command ──
cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_DIR,
    "--served-model-name", MODEL_REPO,
    "--host", "0.0.0.0",
    "--port", "8000",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--tensor-parallel-size", str(TENSOR_PARALLEL),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--trust-remote-code",
    "--dtype", "half",
] + quant_args

print("=" * 60)
print("🚀 STARTING vLLM SERVER (Background)")
print("=" * 60)
print(f"   Model: {MODEL_REPO}")
print(f"   Quantisierung: {quant_label}")
print(f"   Max Context: {MAX_MODEL_LEN}")
print(f"   GPU Utilization: {GPU_MEMORY_UTILIZATION}")
print()

# ── Start as background process ──
log_file = open("/tmp/vllm_server.log", "w")
server_process = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
)
os.environ["VLLM_PID"] = str(server_process.pid)
print(f"   PID: {server_process.pid}")
print(f"   Log: /tmp/vllm_server.log")
print(f"\n⏳ Model wird in GPU-VRAM geladen...\n")

# ── Wait for ready with live status ──
ready = False
for i in range(300):
    try:
        r = requests.get("http://localhost:8000/v1/models", timeout=3)
        if r.status_code == 200:
            ready = True
            break
    except:
        pass

    if i % 10 == 0 and i > 0:
        # Show GPU memory usage during loading
        try:
            mem_output = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
                text=True
            ).strip()
            used, total = mem_output.split(", ")
            pct = int(used) / int(total) * 100
            bar_len = 25
            filled = int(bar_len * pct / 100)
            bar = "█" * filled + "░" * (bar_len - filled)
            print(f"   [{i:>3}s] VRAM [{bar}] {pct:.0f}% ({used}/{total} MB)")
        except:
            print(f"   [{i:>3}s] Loading...")

        # Show latest log line
        try:
            with open("/tmp/vllm_server.log", "r") as f:
                lines = f.readlines()
                for line in lines[-2:]:
                    line = line.strip()
                    if line and ("INFO" in line or "Loading" in line.lower()):
                        print(f"         → {line[-90:]}")
        except:
            pass
    time.sleep(1)

if ready:
    r = requests.get("http://localhost:8000/v1/models")
    models = r.json().get("data", [])
    model_names = ", ".join(m['id'] for m in models)

    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
        text=True
    ).strip()

    display(HTML(
        '<div style="border:2px solid #4CAF50;border-radius:10px;padding:15px;margin:10px 0;background:#1a1a2e;color:#eee;font-family:monospace;">'
        '<div style="font-size:18px;font-weight:bold;color:#4CAF50;margin-bottom:10px;">🚀 vLLM Server ONLINE</div>'
        f'<div style="margin:5px 0;">Model: <b>{model_names}</b></div>'
        f'<div style="margin:5px 0;">Quantisierung: <b>{quant_label}</b></div>'
        f'<div style="margin:5px 0;">API: <b>http://localhost:8000/v1</b></div>'
        f'<div style="margin:5px 0;">VRAM: <b>{gpu_info}</b></div>'
        f'<div style="margin:5px 0;">Ladezeit: <b>~{i}s</b></div>'
        f'<div style="margin:5px 0;">PID: <b>{server_process.pid}</b></div>'
        '</div>'
    ))
else:
    print("\n⚠️ Server nicht ready nach 5 Minuten. Check logs:")
    !tail -20 /tmp/vllm_server.log

In [ ]:
#@title 🌐 5a. Gradio Tunnel — Web UI + Test Chat
import gradio as gr
import requests, os

MODEL_REPO = os.environ.get("MODEL_REPO", "intervitens/mini-magnum-12b-v1.1")

def chat(message, history):
    """Chat interface zum Testen des Models."""
    messages = []
    for turn in history:
        if turn[0]:
            messages.append({"role": "user", "content": turn[0]})
        if turn[1]:
            messages.append({"role": "assistant", "content": turn[1]})
    messages.append({"role": "user", "content": message})

    try:
        resp = requests.post(
            "http://localhost:8000/v1/chat/completions",
            json={
                "model": MODEL_REPO,
                "messages": messages,
                "temperature": 0.8,
                "max_tokens": 512,
                "stream": False,
            },
            timeout=120,
        )
        if resp.status_code == 200:
            return resp.json()["choices"][0]["message"]["content"]
        return f"Error {resp.status_code}: {resp.text[:200]}"
    except Exception as e:
        return f"Error: {e}"

def get_status():
    try:
        r = requests.get("http://localhost:8000/v1/models", timeout=5)
        models = r.json().get("data", [])
        return f"✅ ONLINE | {', '.join(m['id'] for m in models)}"
    except:
        return "❌ OFFLINE"

with gr.Blocks(title="LLM Server", theme=gr.themes.Soft(primary_hue="emerald")) as demo:
    gr.Markdown(f"# 🧠 LLM Server — {MODEL_REPO}")
    with gr.Row():
        status_box = gr.Textbox(label="Server Status", value=get_status(), interactive=False, scale=3)
        refresh_btn = gr.Button("🔄", scale=1)
        refresh_btn.click(fn=get_status, outputs=status_box)
    gr.ChatInterface(fn=chat)

print("🌐 Starte Gradio Tunnel...")
demo.launch(share=True, quiet=False)

In [ ]:
#@title 🌐 5b. Cloudflared Tunnel — SillyTavern API Direktzugang
#@markdown Erstellt einen öffentlichen Tunnel zum vLLM API Port.
#@markdown **Diesen Link in SillyTavern eintragen!**

import subprocess, time, re, os, threading
from IPython.display import display, HTML

MODEL_REPO = os.environ.get("MODEL_REPO", "intervitens/mini-magnum-12b-v1.1")

# Install cloudflared
print("📦 Installing cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb

# Start tunnel
tunnel_log = open("/tmp/cloudflared.log", "w")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.PIPE,
    text=True,
)
print(f"   PID: {tunnel_process.pid}")
print("   Warte auf Tunnel URL...\n")

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_process.stderr.readline()
    if line:
        match = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
        if match:
            tunnel_url = match.group(1)
            break

# Drain stderr
def drain():
    for line in tunnel_process.stderr:
        pass
threading.Thread(target=drain, daemon=True).start()

if tunnel_url:
    os.environ["TUNNEL_URL"] = tunnel_url

    # ── STYLED OUTPUT MASK ──
    display(HTML(f'''
    <div style="border:3px solid #FF6B35;border-radius:12px;padding:20px;margin:15px 0;background:linear-gradient(135deg,#1a1a2e,#16213e);color:#eee;font-family:monospace;">
        <div style="font-size:20px;font-weight:bold;color:#FF6B35;margin-bottom:15px;text-align:center;">
            🔗 SILLYTAVERN CONNECTION
        </div>

        <div style="background:#0d1117;border-radius:8px;padding:15px;margin:10px 0;">
            <div style="color:#58a6ff;font-weight:bold;margin-bottom:8px;">📋 In SillyTavern eintragen:</div>
            <table style="width:100%;border-collapse:collapse;color:#eee;">
                <tr><td style="padding:5px;color:#8b949e;">API Type:</td><td style="padding:5px;"><b>Chat Completion (OpenAI)</b></td></tr>
                <tr><td style="padding:5px;color:#8b949e;">Custom Endpoint:</td><td style="padding:5px;"><code style="background:#161b22;padding:3px 8px;border-radius:4px;color:#7ee787;">{tunnel_url}/v1</code></td></tr>
                <tr><td style="padding:5px;color:#8b949e;">Model:</td><td style="padding:5px;"><code style="background:#161b22;padding:3px 8px;border-radius:4px;color:#7ee787;">{MODEL_REPO}</code></td></tr>
                <tr><td style="padding:5px;color:#8b949e;">API Key:</td><td style="padding:5px;"><i style="color:#8b949e;">(leer lassen)</i></td></tr>
            </table>
        </div>

        <div style="background:#0d1117;border-radius:8px;padding:15px;margin:10px 0;">
            <div style="color:#58a6ff;font-weight:bold;margin-bottom:8px;">🔌 API Endpoints:</div>
            <div style="margin:4px 0;"><code style="color:#7ee787;">{tunnel_url}/v1/chat/completions</code> ← SillyTavern</div>
            <div style="margin:4px 0;"><code style="color:#7ee787;">{tunnel_url}/v1/completions</code> ← Text completion</div>
            <div style="margin:4px 0;"><code style="color:#7ee787;">{tunnel_url}/v1/models</code> ← Model list</div>
            <div style="margin:4px 0;"><code style="color:#d2a8ff;">http://localhost:8000/v1</code> ← Lokal</div>
        </div>

        <div style="text-align:center;margin-top:10px;font-size:11px;color:#8b949e;">
            Tunnel aktiv solange Colab läuft • PID: {tunnel_process.pid}
        </div>
    </div>
    '''))
else:
    print("⚠️ Tunnel URL konnte nicht ermittelt werden.")
    !tail -10 /tmp/cloudflared.log

In [ ]:
#@title 🧪 6. API Test (SillyTavern-kompatibel)
import requests, json, os
from IPython.display import display, HTML

MODEL_REPO = os.environ.get("MODEL_REPO", "intervitens/mini-magnum-12b-v1.1")
tunnel_url = os.environ.get("TUNNEL_URL", None)

results = []

# Test 1: /v1/models
print("[1] GET /v1/models")
try:
    r = requests.get("http://localhost:8000/v1/models", timeout=10)
    if r.status_code == 200:
        models = r.json()["data"]
        results.append(("Models API", "✅", ", ".join(m['id'] for m in models)))
        print(f"    ✅ {len(models)} model(s) gefunden")
    else:
        results.append(("Models API", "❌", f"HTTP {r.status_code}"))
except Exception as e:
    results.append(("Models API", "❌", str(e)[:60]))

# Test 2: /v1/chat/completions
print("\n[2] POST /v1/chat/completions")
try:
    resp = requests.post(
        "http://localhost:8000/v1/chat/completions",
        json={
            "model": MODEL_REPO,
            "messages": [{"role": "user", "content": "Say hello in one sentence."}],
            "temperature": 0.7,
            "max_tokens": 100,
            "stream": False,
        },
        timeout=120,
    )
    if resp.status_code == 200:
        reply = resp.json()["choices"][0]["message"]["content"]
        results.append(("Chat API", "✅", reply[:80]))
        print(f"    ✅ {reply[:100]}")
    else:
        results.append(("Chat API", "❌", f"HTTP {resp.status_code}"))
except Exception as e:
    results.append(("Chat API", "❌", str(e)[:60]))

# Test 3: Tunnel
if tunnel_url:
    print(f"\n[3] Tunnel Test: {tunnel_url}/v1/models")
    try:
        r = requests.get(f"{tunnel_url}/v1/models", timeout=15)
        if r.status_code == 200:
            results.append(("Tunnel", "✅", tunnel_url))
            print(f"    ✅ Tunnel funktioniert!")
        else:
            results.append(("Tunnel", "⚠️", f"HTTP {r.status_code}"))
    except Exception as e:
        results.append(("Tunnel", "❌", str(e)[:60]))

# Result display
rows = ""
for name, status, detail in results:
    color = "#4CAF50" if "✅" in status else "#ff5252"
    rows += f'<tr><td style="padding:5px;">{name}</td><td style="padding:5px;color:{color};">{status}</td><td style="padding:5px;font-size:12px;">{detail}</td></tr>'

display(HTML(f'''
<div style="border:2px solid #2196F3;border-radius:10px;padding:15px;margin:10px 0;background:#1a1a2e;color:#eee;font-family:monospace;">
    <div style="font-size:16px;font-weight:bold;color:#2196F3;margin-bottom:10px;">🧪 API Test Ergebnis</div>
    <table style="width:100%;border-collapse:collapse;color:#eee;">{rows}</table>
</div>
'''))

In [ ]:
#@title 📊 7. Server Status & GPU Monitor
import requests, os, subprocess
from IPython.display import display, HTML

MODEL_REPO = os.environ.get("MODEL_REPO", "intervitens/mini-magnum-12b-v1.1")
tunnel_url = os.environ.get("TUNNEL_URL", None)
vllm_pid = os.environ.get("VLLM_PID", "?")

# Gather status
vllm_status = "❌ OFFLINE"
model_info = ""
try:
    r = requests.get("http://localhost:8000/v1/models", timeout=5)
    models = r.json().get("data", [])
    vllm_status = f"✅ RUNNING ({len(models)} model)"
    model_info = ", ".join(m['id'] for m in models)
except:
    pass

tunnel_status = "❌ Nicht gestartet"
if tunnel_url:
    try:
        r = requests.get(f"{tunnel_url}/v1/models", timeout=10)
        tunnel_status = f"✅ AKTIV"
    except:
        tunnel_status = "⚠️ URL gesetzt, aber nicht erreichbar"

# GPU info
try:
    gpu_out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu",
         "--format=csv,noheader"], text=True
    ).strip()
    gpu_parts = [x.strip() for x in gpu_out.split(",")]
    gpu_name, mem_used, mem_total, gpu_util, gpu_temp = gpu_parts
except:
    gpu_name = mem_used = mem_total = gpu_util = gpu_temp = "?"

display(HTML(f'''
<div style="border:3px solid #9C27B0;border-radius:12px;padding:20px;margin:15px 0;background:linear-gradient(135deg,#1a1a2e,#16213e);color:#eee;font-family:monospace;">
    <div style="font-size:20px;font-weight:bold;color:#9C27B0;margin-bottom:15px;text-align:center;">
        📊 SERVER DASHBOARD
    </div>

    <div style="display:flex;gap:15px;flex-wrap:wrap;">
        <div style="flex:1;min-width:200px;background:#0d1117;border-radius:8px;padding:12px;">
            <div style="color:#58a6ff;font-weight:bold;margin-bottom:8px;">🚀 vLLM Server</div>
            <div>Status: <b>{vllm_status}</b></div>
            <div>Model: <b>{model_info or "—"}</b></div>
            <div>PID: <b>{vllm_pid}</b></div>
            <div>API: <code>localhost:8000/v1</code></div>
        </div>

        <div style="flex:1;min-width:200px;background:#0d1117;border-radius:8px;padding:12px;">
            <div style="color:#58a6ff;font-weight:bold;margin-bottom:8px;">🌐 Tunnel</div>
            <div>Status: <b>{tunnel_status}</b></div>
            <div style="word-break:break-all;">URL: <code style="color:#7ee787;">{tunnel_url or "—"}</code></div>
        </div>

        <div style="flex:1;min-width:200px;background:#0d1117;border-radius:8px;padding:12px;">
            <div style="color:#58a6ff;font-weight:bold;margin-bottom:8px;">🖥️ GPU</div>
            <div>GPU: <b>{gpu_name}</b></div>
            <div>VRAM: <b>{mem_used} / {mem_total}</b></div>
            <div>Auslastung: <b>{gpu_util}</b></div>
            <div>Temperatur: <b>{gpu_temp}</b></div>
        </div>
    </div>
</div>
'''))

In [ ]:
#@title 🔗 8. Connection Info — Kopiere in SillyTavern
import os
from IPython.display import display, HTML

MODEL_REPO = os.environ.get("MODEL_REPO", "intervitens/mini-magnum-12b-v1.1")
tunnel_url = os.environ.get("TUNNEL_URL", "⚠️ Starte Cell 5b für Tunnel")
gpu_name = os.environ.get("GPU_NAME", "?")

display(HTML(f'''
<div style="border:4px solid #FF6B35;border-radius:15px;padding:25px;margin:15px 0;background:linear-gradient(135deg,#0d1117,#161b22);color:#eee;font-family:'Segoe UI',monospace;max-width:700px;">

    <div style="text-align:center;margin-bottom:20px;">
        <div style="font-size:24px;font-weight:bold;color:#FF6B35;">🔗 SILLYTAVERN CONNECTION</div>
        <div style="font-size:12px;color:#8b949e;margin-top:5px;">Kopiere diese Einstellungen in SillyTavern</div>
    </div>

    <div style="background:#1a1a2e;border:1px solid #30363d;border-radius:10px;padding:18px;margin:12px 0;">
        <table style="width:100%;border-collapse:collapse;">
            <tr style="border-bottom:1px solid #21262d;">
                <td style="padding:8px 5px;color:#8b949e;width:140px;">🔧 API Type</td>
                <td style="padding:8px 5px;"><b style="color:#f0f6fc;">Chat Completion (OpenAI)</b></td>
            </tr>
            <tr style="border-bottom:1px solid #21262d;">
                <td style="padding:8px 5px;color:#8b949e;">🌐 Endpoint</td>
                <td style="padding:8px 5px;">
                    <code style="background:#0d1117;color:#7ee787;padding:4px 10px;border-radius:5px;font-size:14px;display:inline-block;">{tunnel_url}/v1</code>
                </td>
            </tr>
            <tr style="border-bottom:1px solid #21262d;">
                <td style="padding:8px 5px;color:#8b949e;">🧠 Model</td>
                <td style="padding:8px 5px;">
                    <code style="background:#0d1117;color:#7ee787;padding:4px 10px;border-radius:5px;font-size:14px;display:inline-block;">{MODEL_REPO}</code>
                </td>
            </tr>
            <tr>
                <td style="padding:8px 5px;color:#8b949e;">🔑 API Key</td>
                <td style="padding:8px 5px;"><i style="color:#6e7681;">(leer lassen)</i></td>
            </tr>
        </table>
    </div>

    <div style="background:#1a1a2e;border:1px solid #30363d;border-radius:10px;padding:15px;margin:12px 0;">
        <div style="color:#58a6ff;font-weight:bold;margin-bottom:10px;">📡 API Endpoints</div>
        <div style="margin:5px 0;font-size:13px;">
            <span style="color:#f97583;">POST</span> <code style="color:#7ee787;">{tunnel_url}/v1/chat/completions</code>
        </div>
        <div style="margin:5px 0;font-size:13px;">
            <span style="color:#f97583;">POST</span> <code style="color:#7ee787;">{tunnel_url}/v1/completions</code>
        </div>
        <div style="margin:5px 0;font-size:13px;">
            <span style="color:#d2a8ff;">GET&nbsp;</span> <code style="color:#7ee787;">{tunnel_url}/v1/models</code>
        </div>
    </div>

    <div style="text-align:center;margin-top:15px;font-size:11px;color:#484f58;">
        🖥️ {gpu_name} • 🧠 {MODEL_REPO} • ⚡ vLLM Engine
    </div>
</div>
'''))

In [ ]:
#@title ⏳ 9. Keep Alive — Verhindert Colab Disconnect
#@markdown Läuft im Hintergrund und pingt den Server alle 60 Sekunden.
import time, requests, os
from IPython.display import display, HTML, clear_output

MODEL_REPO = os.environ.get("MODEL_REPO", "intervitens/mini-magnum-12b-v1.1")
tunnel_url = os.environ.get("TUNNEL_URL", "N/A")

print("⏳ Keep-Alive aktiv. Drücke STOP zum Beenden.\n")

counter = 0
while True:
    counter += 1
    try:
        r = requests.get("http://localhost:8000/v1/models", timeout=5)
        models = len(r.json().get("data", []))
        status = f"✅ vLLM OK ({models} model)"
    except:
        status = "❌ vLLM DOWN"

    tunnel_ok = ""
    if tunnel_url != "N/A":
        try:
            requests.get(f"{tunnel_url}/v1/models", timeout=10)
            tunnel_ok = "✅"
        except:
            tunnel_ok = "⚠️"

    ts = time.strftime("%H:%M:%S")
    print(f"\r[{ts}] #{counter:>4} | {status} | Tunnel: {tunnel_ok} {tunnel_url[:50]}   ", end="", flush=True)
    time.sleep(60)

In [ ]:
#@title 📥 10. Weiteres Model herunterladen (Optional)
#@markdown Lade ein weiteres HuggingFace Model herunter.
EXTRA_MODEL_REPO = "" #@param {type:"string"}

from huggingface_hub import snapshot_download
from pathlib import Path
import os, time
from IPython.display import display, HTML

if EXTRA_MODEL_REPO.strip():
    repo = EXTRA_MODEL_REPO.strip()

    use_gdrive = os.environ.get("USE_GDRIVE", "0") == "1"
    gdrive_path = os.environ.get("GDRIVE_MODEL_PATH", "/content/drive/MyDrive/LLM_Models")

    if use_gdrive:
        model_dir = os.path.join(gdrive_path, repo.replace('/', '--'))
    else:
        model_dir = f"/root/models/{repo.replace('/', '--')}"

    print(f"📥 Downloading {repo}...")
    print(f"   → {model_dir}\n")
    start = time.time()

    local_path = snapshot_download(
        repo_id=repo,
        local_dir=model_dir,
        token=os.environ.get("HF_TOKEN"),
        resume_download=True,
    )

    elapsed = time.time() - start
    final_size = sum(f.stat().st_size for f in Path(local_path).rglob('*') if f.is_file()) / (1024**3)

    display(HTML(
        f'<div style="border:2px solid #4CAF50;border-radius:10px;padding:15px;margin:10px 0;background:#1a1a2e;color:#eee;font-family:monospace;">'
        f'<div style="font-size:16px;font-weight:bold;color:#4CAF50;">✅ {repo} heruntergeladen!</div>'
        f'<div style="margin:5px 0;">Größe: <b>{final_size:.1f} GB</b> | Zeit: <b>{elapsed:.0f}s</b></div>'
        f'<div style="margin:5px 0;">Pfad: <code>{local_path}</code></div>'
        f'<div style="margin:8px 0;color:#FFC107;">ℹ️ Um dieses Model zu laden: Cell 4 neustarten mit MODEL_DIR=\'{model_dir}\'</div>'
        f'</div>'
    ))
else:
    print("ℹ️ Model Repo oben eintragen (z.B. 'mistralai/Mistral-7B-Instruct-v0.3')")

In [ ]:
#@title 🚨 11. Server Logs (Debug)
print("═" * 60)
print("  📜 vLLM SERVER LOG (letzte 30 Zeilen)")
print("═" * 60)
!tail -30 /tmp/vllm_server.log 2>/dev/null || echo "Kein Log gefunden"
print()
print("═" * 60)
print("  📜 CLOUDFLARED LOG (letzte 10 Zeilen)")
print("═" * 60)
!tail -10 /tmp/cloudflared.log 2>/dev/null || echo "Kein Log gefunden"